# UD2: Lenguajes de Marcas y Web Scraping

Bienvenido a la Unidad 2. En este cuaderno aprenderemos a manejar datos estructurados (**JSON**) y a extraer información de páginas web (**Web Scraping**) de manera automática.

## Índice
1.  **JSON**: El lenguaje de intercambio de datos.
2.  **Web Scraping Básico**: `requests` y `BeautifulSoup`.
3.  **Automatización**: `selenium`.
4.  **Ejercicio Final**: Fotocasa con Selenium.
5.  **Moderno**: `playwright`.

---

## Parte 1: JSON (JavaScript Object Notation)

**JSON** es un formato de texto ligero para el intercambio de datos. Es muy fácil de leer para los humanos y fácil de analizar para las máquinas. Se basa en dos estructuras:
*   **Objetos**: Colecciones de pares clave/valor (como los diccionarios en Python). `{ "clave": "valor" }`
*   **Arreglos (Arrays)**: Listas ordenadas de valores. `[ "valor1", "valor2" ]`

En Python, usamos la librería estándar `json`.

In [ ]:
import json 

# Ejemplo de una cadena texto en JSON en texto
json_str = '{"nombre":"R2-D2, "tipo":"Droide", "altura":1}'

# Pasar de JSON a Python ---> Deserialización
datos = json.load(json_str)
print("Tipo de dato: ", type(datos))
print("Nombre: ", datos["nombre"])

# Pasar de Python a JSON ---> Serializacion

droide = {
    "nombre":"C3P0",
    "tipo":"Protocolo",
    "altura":2
}

json_output = json.dumps(droide, indent=4)
print(json_output)

Tipo de dato: <class 'dict'>
Nombre: R2-D2

JSON generado:
 {
    "nombre": "C-3PO",
    "tipo": "Protocolo",
    "idiomas": 6000000
}


### Trabajando con ficheros JSON
Normalmente leeremos y escribiremos en archivos `.json`.

In [ ]:
from pathlib import Path
import json 

# Establecemos la ruta al fichero 
ruta = Path("data/flota.json")

# Leer el fichero existente 

if ruta.exists():
    with open(ruta, "r", encoding="utf-8") as archivito:
        flota = json.load(archivito)
        
    print(f"Fichero cargado. Hay {len(flota["flota"])} naves registradas")
    
else:
    print("No existe")

Fichero cargado. Hay 4 naves registradas


### 📝 Examen de Mentira 1: Gestión de Flota

**Objetivo**: Cargar el fichero `data/flota.json`, filtrar solo las naves que estén en estado "Activo" y guardar el resultado en un nuevo fichero llamado `data/flota_activa.json`.

1.  Carga el JSON.
2.  Recorre la lista de naves.
3.  Guarda en una lista nueva solo las activas.
4.  Escribe esa lista en un nuevo fichero JSON.

¡Hecho! Se han guardado 2 naves activas en data\flota_activa.json


---

## Parte 2: Web Scraping Básico

El Web Scraping consiste en descargar el código HTML de una web y extraer información útil.

**Herramientas básicas:**
*   **`requests`**: Para hacer la petición HTTP (como si fueras el navegador) y descargar el HTML.
*   **`BeautifulSoup` (bs4)**: Para "parsear" (analizar) el HTML y buscar etiquetas (`<div>`, `<h1>`, etc.).

Vamos a practicar con nuestra propia web local de naves espaciales.

In [ ]:
# !pip install requests beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [beautifulsoup4]


In [7]:
import requests 
from bs4 import BeautifulSoup
from pathlib import Path
import os

# Nota: En este ejemplo como la web es un fichero local lo leemos directamente
# En una web real: respuesta = request.get("http://")

ruta = Path("web_naves/index.html")

if ruta.exists():
    with open(ruta, "r", encoding="utf-8") as web:
        html = web.read()
        
    # 1. Formar la sopa 
    soup = BeautifulSoup(html, "html.parser") 
    #print(soup.prettify()[:500], "...\n") # 
    
    # 2. Obtener un elemento del DOM
    titulo = soup.title.text
    print(f"Titulo de la web: {titulo}")
    
    # 3. Obtener todos los nombres de las naves <h3>
    nombres = soup.find_all("h3")
    for n in nombres:
        print("- ", n.text)
    



Titulo de la web: Concesionario Galáctico | Naves Premium
-  🔥 Oferta del Mes: Caza 'Furia' MK-II
-  Exploradora X-1
-  Caza 'Furia' MK-II
-  Carguero 'Mula' T-500
-  Velocista S-9
-  Nave 'Pinky' Z-1
-  Ubicación
-  Comunicaciones


### Selectores CSS
BeautifulSoup permite buscar usando selectores CSS (como en la hoja de estilos) con `.select()`.
*   `.clase` busca por clase.
*   `#id` busca por ID.
*   `etiqueta` busca por etiqueta.

In [12]:
precios = soup.select(".precio") # Seleccionamos los elementos de una clase 

for p in precios:
    print(p.text)

ofertas = soup.select("#ofertas") # Seleccionamos el elemento con un id
print(ofertas)

1,500,000 cr
2,300,000 cr
850,000 cr
3,000,000 cr
999,999 cr
[<section class="offers-section" id="ofertas">
<div class="container">
<div class="offer-banner">
<h3>🔥 Oferta del Mes: Caza 'Furia' MK-II</h3>
<p>¡Descuento del 15% si pagas en Créditos Imperiales!</p>
</div>
</div>
</section>]


### Descargando Imágenes con BeautifulSoup

A menudo querremos descargar las imágenes de un catálogo. El proceso es:
1.  Encontrar la etiqueta `<img>`.
2.  Obtener el atributo `src` (la URL de la imagen).
3.  Usar `requests.get(url_imagen)` para descargar los bytes.
4.  Guardar esos bytes en un archivo local.

In [ ]:
import os
from pathlib import Path

# 1. Crear un directorio para las descargas 
os.makedirs("descargas_img_2", exist_ok=True)

# 2. Buscar todas las imagenes 
imagenes = soup.find_all("img")
print(f"Encontradas: {len(imagenes)}.Descargando...")

# 3. Vamos a recorrer todas las imagenes encontradas 
for i, img in enumerate(imagenes):
    # Obtener su URL
    src = img.get("src")
    
    nombre_archivo = f"nave_{i+1}.png"
    ruta_destini = os.path.join("descargas_img_2", nombre_archivo)
    
    # Guardar la imagen
    
    try:
        with open(src, "r" ) as entrada:
            datos = entrada.read()

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

### 📝 Examen de Mentira 2: Escrapeando el Concesionario

**Objetivo**: Extraer una lista de diccionarios con la información de cada nave de `web_naves/index.html`.

Estructura deseada para cada nave:
```python
{
  "nombre": "...",
  "precio": "...",
  "descripcion": "..."
}
```

1.  Usa `soup.find_all("article", class_="nave-card")` para obtener las tarjetas de las naves.
2.  Itera sobre cada tarjeta.
3.  Dentro de cada tarjeta, busca el `h3` (nombre), el `.precio` y la `.descripcion`.
4.  Guárdalo en una lista e imprímela.

Naves extraídas:
{'nombre': 'Exploradora X-1', 'precio': '1,500,000 cr', 'descripcion': 'Ideal para perderse en el borde exterior. Incluye cafetera cuántica y\n                        asientos de piel de Wookiee sintética.'}
{'nombre': "Caza 'Furia' MK-II", 'precio': '2,300,000 cr', 'descripcion': 'Rápida, letal y roja. Porque el rojo corre más. No incluye seguro de vida.'}
{'nombre': "Carguero 'Mula' T-500", 'precio': '850,000 cr', 'descripcion': 'Fea pero robusta. Carga hasta 500 toneladas de mineral o mercancía "no\n                        preguntes".'}
{'nombre': 'Velocista S-9', 'precio': '3,000,000 cr', 'descripcion': 'Diseñada para ganar. Si no ganas, es culpa del piloto.'}
{'nombre': "Nave 'Pinky' Z-1", 'precio': '999,999 cr', 'descripcion': 'La nave más adorable de la galaxia. Dispara arcoíris (metafóricamente).'}


---

## Parte 3: Automatización de Navegadores con Selenium

Cuando una web depende mucho de JavaScript (botones que cargan cosas, popups, scroll infinito) o necesitamos interactuar con ella (hacer login, rellenar formularios), `requests` se queda corto. Aquí entra **Selenium**.

**Selenium** permite controlar un navegador web real (Chrome, Firefox, Edge) mediante código. Es como tener un robot que navega por ti.

### Métodos Importantes
*   **`driver.get(url)`**: Navegar a una página.
*   **`driver.find_element(By.ID, "...")`**: Buscar un elemento (uno solo).
*   **`driver.find_elements(By.CLASS_NAME, "...")`**: Buscar varios elementos (lista).
*   **`elemento.click()`**: Hacer clic.
*   **`elemento.send_keys("texto")`**: Escribir en un input.
*   **`elemento.text`**: Obtener el texto del elemento.

**Instalación**: `pip install selenium`

In [ ]:
# !pip install selenium


Título: Concesionario Galáctico | Naves Premium
Haciendo clic en siguiente...
Nueva página: Concesionario Galáctico | Modelos Exclusivos


### 📝 Examen de Mentira 3: Scraping Completo con Navegación

**Objetivo**: Usar Selenium para recorrer **todas** las páginas del concesionario de naves y guardar **todas** las naves en un CSV.

1.  Empieza en `index.html`.
2.  Bucle: Extrae las naves de la página actual.
3.  Busca el botón "Siguiente Página →".
4.  Si existe, haz clic y repite. Si no, termina.
5.  Exporta la lista final a `data/todas_las_naves_selenium.csv`.

Escrapeando página... encontradas 5 naves.
Escrapeando página... encontradas 3 naves.
Escrapeando página... encontradas 2 naves.
No hay más páginas o botón no encontrado. Fin.
Total naves: 10
                  Nombre        Precio  \
0        Exploradora X-1  1,500,000 cr   
1     Caza 'Furia' MK-II  2,300,000 cr   
2  Carguero 'Mula' T-500    850,000 cr   
3          Velocista S-9  3,000,000 cr   
4       Nave 'Pinky' Z-1    999,999 cr   
5    Yate Estelar 'Lujo'  5,000,000 cr   
6     Interceptor 'Rayo'  1,800,000 cr   
7        Minero 'Topo' X    900,000 cr   
8    El 'Halcón' (Copia)        500 cr   
9            Sonda 'Ojo'    150,000 cr   

                                         Descripcion  
0  Ideal para perderse en el borde exterior. Incl...  
1  Rápida, letal y roja. Porque el rojo corre más...  
2  Fea pero robusta. Carga hasta 500 toneladas de...  
3  Diseñada para ganar. Si no ganas, es culpa del...  
4  La nave más adorable de la galaxia. Dispara ar...  
5  Para el dict

## Parte 4: Web Scraping Moderno con Playwright

**Selenium** es genial, pero a veces es lento y difícil de configurar. La industria está moviéndose hacia herramientas más modernas como **Playwright**.

### ¿Qué es Playwright?
Es una librería de automatización creada por Microsoft. Es **más rápida**, **más fiable** y capaz de manejar webs muy complejas (Single Page Applications) mejor que Selenium.

### Conceptos Clave
1.  **Chromium**: Es el motor de navegador de código abierto en el que se basa Google Chrome y Microsoft Edge. Playwright usa una versión ligera de Chromium para navegar.
2.  **Asyncio (Asincronía)**: Playwright es asíncrono. Esto significa que mientras espera a que una web cargue, tu programa puede hacer otras cosas. Usamos `await` para decir "espera aquí hasta que esto termine, pero no bloquees todo el ordenador".
3.  **Contextos**: Puedes tener varias "sesiones" de navegador aisladas (como perfiles distintos) en una sola instancia.

**Instalación**: `pip install playwright` y `playwright install`

In [52]:
# !pip install playwright
# !playwright install


### Async y Await   

`async`: Define una función que puede ser pausada (una "corrutina").

`await`: Pausa la ejecución de la función hasta que se resuelve la promesa.

In [ ]:
#!pip install asyncio
import asyncio
import time


 [Abrir script webscraping con playwright](UD2_WebScraping_Parte_4.py)

### 📝 Ejercicio Final: Detective de Noticias

**Objetivo**: Crear un script que vigile la portada de un periódico digital (ej: `elpais.com`, `elmundo.es`, o `marca.com`) y detecte si aparecen ciertas palabras clave en los titulares.

**Instrucciones**:
1.  Usa `Playwright` para abrir la web.
2.  Busca todos los titulares (inspecciona la web para ver qué etiqueta usan, suele ser `h2`, `h3` o `article`).
3.  Recorre los textos y busca si contienen palabras como "Crisis", "Fútbol", "Gobierno" o lo que prefieras.
4.  Imprime "¡ALERTA! Noticia encontrada: [Titular]" si hay coincidencia.

In [ ]:
# ¡OJO! Ejecutar en un fichero .py ya que en Jupyter no funciona 

# Plantilla (opcional):

import 

# --- FIX PARA WINDOWS ---
# Mantenemos esto siempre para asegurar que funciona bien en Windows
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# 1. FUNCIÓN PRINCIPAL ASÍNCRONA
async def main():
    
    # PALABRAS CLAVE: Lo que queremos buscar (puedes cambiarlas)
    palabras_clave = []
    print(f"📋 Buscando: {', '.join(palabras_clave)}")

    async with async_playwright() as p:
        
        # 2. LANZAMIENTO
        # Usamos headless=False para ver cómo el 'detective' abre el periódico
        

        # 3. NAVEGACIÓN
        target_url = 
        print(f"🌍 Abriendo la portada de: {target_url}")
        
        # wait_until='domcontentloaded': Espera a que esté el texto listo (es más rápido que esperar a todas las fotos)

        # 4. GESTIÓN DE COOKIES (Opcional pero recomendado)
        # Intentamos cerrar el banner para que no moleste, aunque para leer texto a veces no hace falta.


        # 5. EXTRACCIÓN MASIVA DE TITULARES
        print("🔍 Escaneando titulares...")
        
        # page.locator: Le decimos que busque TODAS las etiquetas de título o con algún selector en particular
        # .all_inner_texts(): Esta función es MÁGICA. En lugar de hacer un bucle manual,
        # nos devuelve directamente una LISTA con el texto de todos los elementos encontrados.

        
        print(f"📉 Se han encontrado {len(titulares)} posibles titulares. Analizando...")
        
        # 6. LÓGICA DE DETECTIVE (Filtrado)
        noticias_encontradas = 0
        
        for texto in titulares:
            # Limpiamos espacios en blanco al principio y final
            
            
            # Si el titular está vacío, pasamos al siguiente
                
            # Comprobamos si alguna palabra clave está dentro del texto
                    # Usamos break para no imprimir la misma noticia 2 veces si tiene dos palabras clave
                    break 

        if noticias_encontradas == 0:
            print("😴 Todo tranquilo. No se encontraron las palabras clave en la portada.")
        else:
            print(f"✅ Análisis completado. Total alertas: {noticias_encontradas}")

        # 7. CIERRE
        

# PUNTO DE ARRANQUE DEL SCRIPT
if __name__ == "__main__":
    asyncio.run(main())

